# M1 final — severity of the CVEs without a CVSS score (US10, #18)

**Question (FR-23):** how well can the CVSS severity class of a CVE be predicted from its record, for the CVEs that get no score (risk R1)?

Sprint 1: text only, macro-F1 **0.516** (NO-GO, risk R4 plan); with the affected products, **0.521**. This notebook explains the final model of Sprint 3.

| CRISP-DM step | In this notebook |
|---|---|
| Business understanding | CVEs without any CVSS score, by year |
| Data understanding | Do the organizations that score CVEs agree with each other? |
| Data preparation | Labels: CVSS 3.x score of the CNA or of an independent analyst (CISA ADP, else NVD). Features: what the CNA publishes in the record |
| Modeling | TF-IDF + logistic regression. Feature sets compared on the validation period; a second view predicts the 8 CVSS base metrics and applies the official CVSS formula |
| Evaluation | Version trained by `train_m1`: macro-F1, F1 by class, confusion matrix on the test period; strict split and model retrained each quarter |
| Deployment | Predictions stored by the pipeline (scoring step, UC19) |

The training code is `backend/ml/m1.py`, shared with the command `train_m1` (FR-26: each version and its metrics are registered in `ml_mlmodel`).

**Prerequisites:** Docker running; CVEs reloaded once with `run_pipeline --source cve --full` (CNA and title of each CVE), NVD loaded; `train_m1 --activate` then `score_m1` done; virtual environment `.venv` with `requirements-dev.txt`.

## 1. Connection and shared code

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import psycopg

def find_repo_root(start=Path.cwd()):
    for folder in [start, *start.parents]:
        if (folder / "docker-compose.yml").exists():
            return folder
    raise FileNotFoundError("Run the notebook from inside the threatlens repository")

def read_env(path):
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "backend"))
from ml import m1  # same code as the train_m1 command

env = read_env(ROOT / ".env")
conn = psycopg.connect(host="localhost", port=int(env.get("POSTGRES_PORT", 5432)),
                       dbname=env["POSTGRES_DB"], user=env["POSTGRES_USER"], password=env["POSTGRES_PASSWORD"])
query = lambda sql, params=None: m1.query(conn, sql, params)
pd.set_option("display.max_colwidth", 80)
print("Connected to", env["POSTGRES_DB"])

Connected to threatlens


## 2. Business understanding: CVEs without any CVSS score

These are the CVEs M1 predicts: neither the NVD, nor the CNA, nor CISA ADP gave a score (any CVSS version).

In [2]:
unscored = query("""
    SELECT extract(year FROM v.published_at)::int AS year,
           count(*) AS cves,
           count(*) FILTER (WHERE NOT EXISTS (
               SELECT 1 FROM cti_cvssmetric m WHERE m.vulnerability_id = v.id)) AS without_cvss
    FROM cti_vulnerability v
    WHERE v.published_at >= %s
    GROUP BY 1 ORDER BY 1
""", [m1.SINCE])
unscored["share_without_cvss"] = (unscored.without_cvss / unscored.cves).round(3)
unscored.set_index("year")

,cves,without_cvss,share_without_cvss
year,,,
2018,16510,0,0.000
2019,17308,0,0.000
2020,18363,0,0.000
2021,20177,0,0.000
2022,24999,2,0.000
2023,28847,1,0.000
2024,39924,8,0.000
2025,48152,897,0.019
2026,97740,3218,0.033


In [3]:
query("""
    SELECT v.assigner AS cna, count(*) AS cves_without_cvss
    FROM cti_vulnerability v
    WHERE v.published_at >= %s AND NOT EXISTS (SELECT 1 FROM cti_cvssmetric m WHERE m.vulnerability_id = v.id)
    GROUP BY 1 ORDER BY 2 DESC LIMIT 8
""", [m1.TEST_FROM]).set_index("cna")

,cves_without_cvss
cna,
linux,3924
apache,61
mozilla,38
wpscan,17
cpansec,12
go,12
certcc,11
devolutions,7


## 3. Data understanding: do the scorers agree?

A model learns from the scores given by people. Before asking a model to reach a macro-F1 of 0.70, we measure how well two organizations agree when they score **the same CVE** with CVSS 3.x: `same_class` is the share of CVEs where both give the same severity class, `macro_f1` reads the first one as a prediction of the second.

In [4]:
m1.agreement(conn).set_index(["period", "providers"])

cves  same_class  macro_f1
period                         providers                                   
since 2018-01-01               CISA_ADP vs CNA   1594       0.679     0.582
                               CISA_ADP vs NVD  18348       0.938     0.896
                               CNA vs NVD       72598       0.567     0.460
since 2025-07-01 (test period) CISA_ADP vs CNA    607       0.852     0.721
                               CISA_ADP vs NVD   1549       0.866     0.797
                               CNA vs NVD       12172       0.430     0.345

**Reading.** CISA ADP and the NVD almost always agree: they are independent analysts with the same method. The CNA (the vendor of the product) and the NVD often give different classes for the same CVE. The same description therefore has two valid answers, depending on who scores.

**Consequences for M1:**

- A label has a **scorer**: `cna` (the CNA scored its own CVE) or `analyst` (CISA ADP, else the NVD). M1 learns both and is told which one to imitate. A CVE scored by both gives two training rows.
- A CVE without any score was not scored by its CNA: in production M1 predicts the score **an analyst would give**.
- Evaluation: one row per CVE, the score of its CNA when there is one, else the analyst's.

## 4. Data preparation

In [5]:
data = m1.load_labelled(conn)
main = data[data.main]  # one row per CVE: used for the evaluation
print(f"{len(main):,} labelled CVEs since {m1.SINCE} ({len(data):,} training rows: {len(data) - len(main):,} CVEs are scored by their CNA and by an analyst)")
pd.crosstab(main.published_at.dt.year, main.label_provider)

299,341 labelled CVEs since 2018-01-01 (372,839 training rows: 73,498 CVEs are scored by their CNA and by an analyst)


label_provider,CISA_ADP,CNA,NVD
published_at,,,
2018,190,1554,14766
2019,228,2414,14666
2020,254,5826,12283
2021,345,8039,11793
2022,4089,10683,10214
2023,5832,17318,5696
2024,9808,26915,2875
2025,8642,32066,4679
2026,15353,65823,6990


In [6]:
pd.Series({
    "with a title": (main.title != "").mean(),
    "with at least one affected product": (main.products != "").mean(),
    "with a CWE named by the CNA": (main.cwes != "").mean(),
    "with the CNA name": (main.assigner != "").mean(),
}).round(3).to_frame("share of labelled CVEs")

,share of labelled CVEs
with a title,0.556
with at least one affected product,0.736
with a CWE named by the CNA,0.562
with the CNA name,1.000


In [7]:
fit, valid, test = m1.split(data)
valid, test = valid[valid.main], test[test.main]
pd.DataFrame({
    "period": [f"{m1.SINCE} to {m1.VALID_FROM}", f"{m1.VALID_FROM} to {m1.TEST_FROM}", f"from {m1.TEST_FROM}"],
    "CVEs": [fit.cve_id.nunique(), len(valid), len(test)],
    **{name: [round((part.label == name).mean(), 3) for part in (fit[fit.main], valid, test)] for name in m1.CLASSES},
}, index=["fit", "validation", "test"])

,period,CVEs,LOW,MEDIUM,HIGH,CRITICAL
fit,2018-01-01 to 2024-07-01,146500,0.036,0.442,0.390,0.132
validation,2024-07-01 to 2025-07-01,42288,0.044,0.498,0.368,0.090
test,from 2025-07-01,110553,0.044,0.429,0.421,0.106


## 5. Modeling

### 5.1 Which features help? (validation period only)

Each feature set is trained on the fit period and scored on the validation period, with the direct view (one classifier for the severity class). The test period is not used here.

In [8]:
rows = []
for name, features in m1.FEATURE_SETS.items():
    model = m1.SeverityModel(features, with_metrics=False).fit(fit)
    rows.append({"features": name, "validation macro-F1": m1.macro_f1(valid.label, model.predict(valid)[0])})
    print(rows[-1])
pd.DataFrame(rows).set_index("features")

{'features': 'text only', 'validation macro-F1': 0.529}
{'features': 'text + products', 'validation macro-F1': 0.545}
{'features': 'text + products + CWE + CNA', 'validation macro-F1': 0.531}
{'features': 'text + products + CWE + CNA + title', 'validation macro-F1': 0.563}
{'features': 'text + products + CWE + CNA + title + scorer and its words', 'validation macro-F1': 0.617}


,validation macro-F1
features,
text only,0.529
text + products,0.545
text + products + CWE + CNA,0.531
text + products + CWE + CNA + title,0.563
text + products + CWE + CNA + title + scorer and its words,0.617


### 5.2 Second view: the CVSS base metrics

A CVSS score is computed from 8 base metrics (attack vector, attack complexity, privileges required, user interaction, scope, and the impacts on confidentiality, integrity and availability). Words such as *remote*, *unauthenticated* or *denial of service* describe these metrics more directly than the final class. The second view therefore trains one logistic regression per base metric and applies the **official CVSS formula** (`backend/ml/cvss.py`) to get the probability of each class.

`train_m1` combines both views; the share of the metrics view is chosen on the validation period (0 = direct view only, 1 = metrics view only). The results below are read from the registered version.

In [9]:
version = query("""
    SELECT name, version, algorithm, metrics, trained_at, is_active
    FROM ml_mlmodel WHERE task = 'SEVERITY' ORDER BY is_active DESC, trained_at DESC LIMIT 1
""").iloc[0]
metrics = version.metrics if isinstance(version.metrics, dict) else json.loads(version.metrics)
print(f"{version['name']} v{version.version} ({'active' if version.is_active else 'not active'}): {version.algorithm}")
pd.Series(metrics["validation_macro_f1_by_metric_weight"]).rename_axis("share of the metrics view").to_frame("validation macro-F1")

m1-severity v1 (active): TF-IDF + logistic regression: severity class and CVSS base metrics (official CVSS formula), combined


,validation macro-F1
share of the metrics view,
0.0,0.616
0.5,0.650
0.7,0.656
1.0,0.636


## 6. Evaluation on the test period

Two measures, both with a time-based split:

- **strict split**: one model trained before the test period predicts all of it, without ever being retrained;
- **retrained each quarter**: each period of 3 months is predicted by a model trained on the CVEs published before it. This is how M1 is meant to be used (a new version each quarter, FR-26).

In [10]:
quarterly = metrics.get("retrained_each_quarter", {})
pd.Series(dtype=object, data={
    "test macro-F1, strict split": metrics["macro_f1"],
    "test macro-F1, retrained each quarter": quarterly.get("macro_f1"),
    "target (FR-23)": metrics["target_macro_f1"],
    "accuracy (strict split)": metrics["accuracy"],
    "right or one class away (strict split)": metrics["right_or_one_class_away"],
    "reference: description only, same split": metrics["reference_text_only"]["macro_f1"],
    "share of the metrics view": metrics["metric_weight"],
    "test CVEs": metrics["sizes"]["test"],
}).to_frame(f"{version['name']} v{version.version}")

,m1-severity v1
"test macro-F1, strict split",0.571
"test macro-F1, retrained each quarter",0.605
target (FR-23),0.7
accuracy (strict split),0.673
right or one class away (strict split),0.968
"reference: description only, same split",0.501
share of the metrics view,0.7
test CVEs,110553


In [11]:
pd.Series(metrics["f1_by_class"]).reindex(m1.CLASSES).to_frame("F1 on the test period (strict split)").T

,LOW,MEDIUM,HIGH,CRITICAL
F1 on the test period (strict split),0.371,0.73,0.677,0.507


In [12]:
matrix = metrics["confusion_matrix"]
pd.DataFrame(matrix["counts"], index=[f"true {c}" for c in matrix["classes"]],
             columns=[f"pred {c}" for c in matrix["classes"]])

,pred LOW,pred MEDIUM,pred HIGH,pred CRITICAL
true LOW,1283,2809,650,104
true MEDIUM,686,35710,9598,1478
true HIGH,97,10618,31336,4469
true CRITICAL,5,1214,4460,6036


**By origin of the label.** *Not scored by the CNA* are the CVEs that look most like those M1 predicts in production.

In [13]:
pd.DataFrame(metrics["by_scorer"]).T

,cves,accuracy,macro_f1
label of the NVD,9743.0,0.716,0.545
label of CISA ADP,19405.0,0.610,0.457
scored by the CNA,81405.0,0.683,0.587
not scored by the CNA,29148.0,0.645,0.482


**Quarter by quarter.** New CNAs appear and scoring habits change: a model that is not retrained loses quality.

In [14]:
pd.DataFrame(quarterly.get("quarters", [])).rename(columns={"from": "quarter from", "trained_on": "trained on (CVEs)"})

,cves,quarter from,accuracy,macro_f1,trained on (CVEs)
0,11037,2025-07-01,0.725,0.664,188788
1,11350,2025-10-01,0.741,0.642,199825
2,14596,2026-01-01,0.705,0.625,211175
3,19622,2026-04-01,0.680,0.581,225771
4,32244,2026-07-01,0.684,0.579,245393
5,21704,2026-10-01,0.700,0.585,277637


**Is the confidence shown to the analyst useful?** Scores of the most confident predictions (strict split).

In [15]:
pd.DataFrame(metrics["by_confidence"]).rename(columns={"share": "most confident share of the predictions"})

,most confident share of the predictions,accuracy,macro_f1,min_confidence
0,1.0,0.673,0.571,0.32
1,0.8,0.727,0.628,0.54
2,0.6,0.786,0.696,0.66
3,0.4,0.852,0.778,0.80


**Comparison with Sprint 1** (same test period; the Sprint 1 labels were the CNA's score, else CISA ADP's).

In [16]:
baseline = json.loads((ROOT / "ml" / "models" / "m1_baseline_metrics.json").read_text())
improvement = json.loads((ROOT / "ml" / "models" / "m1_improvement_metrics.json").read_text())
pd.DataFrame([
    {"model": "Sprint 1 baseline: description only", "test macro-F1": baseline["macro_f1"]},
    {"model": "Sprint 1: description + products", "test macro-F1": improvement["macro_f1"]},
    {"model": "Sprint 3 reference: description only, labels of this notebook", "test macro-F1": metrics["reference_text_only"]["macro_f1"]},
    {"model": f"Sprint 3 final M1 v{version.version}, strict split", "test macro-F1": metrics["macro_f1"]},
    {"model": f"Sprint 3 final M1 v{version.version}, retrained each quarter", "test macro-F1": quarterly.get("macro_f1")},
]).set_index("model")

,test macro-F1
model,
Sprint 1 baseline: description only,0.516
Sprint 1: description + products,0.521
"Sprint 3 reference: description only, labels of this notebook",0.501
"Sprint 3 final M1 v1, strict split",0.571
"Sprint 3 final M1 v1, retrained each quarter",0.605


## 7. Deployment: predictions stored by the pipeline

In [17]:
query("""
    SELECT p.predicted_severity, count(*) AS cves, round(avg(p.confidence)::numeric, 3) AS mean_confidence,
           count(*) FILTER (WHERE p.confidence < 0.5) AS below_50_percent
    FROM ml_severityprediction p
    JOIN ml_mlmodel m ON m.id = p.model_id AND m.is_active
    JOIN cti_vulnerability v ON v.id = p.vulnerability_id AND v.is_severity_predicted
    GROUP BY p.predicted_severity
""").set_index("predicted_severity").reindex(m1.CLASSES)

,cves,mean_confidence,below_50_percent
predicted_severity,,,
LOW,NaN,NaN,NaN
MEDIUM,3106.0,0.822,16.0
HIGH,975.0,0.749,30.0
CRITICAL,45.0,0.717,4.0


## 8. Result

In [18]:
agreement = m1.agreement(conn)
cna_nvd = agreement[(agreement.providers == "CNA vs NVD") & agreement.period.str.startswith("since " + m1.SINCE)]
not_cna = metrics["by_scorer"].get("not scored by the CNA", {})
print(f"M1 v{version.version}, test period {metrics['periods']['test']} ({metrics['sizes']['test']:,} CVEs):")
print(f"- macro-F1 {metrics['macro_f1']} with the strict split" + (f", {quarterly['macro_f1']} retrained each quarter" if quarterly else "")
      + f" (description only: {metrics['reference_text_only']['macro_f1']}; indicative target: {metrics['target_macro_f1']})")
print(f"- accuracy {metrics['accuracy']:.1%}; right or one class away {metrics['right_or_one_class_away']:.1%}")
print(f"- CVEs not scored by their CNA (those M1 predicts): macro-F1 {not_cna.get('macro_f1')}, accuracy {not_cna.get('accuracy')}")
if len(cna_nvd):
    print(f"- for comparison, the CNA and the NVD scoring the same CVE: same class for {cna_nvd.same_class.iloc[0]:.1%} "
          f"of {cna_nvd.cves.iloc[0]:,} CVEs, macro-F1 {cna_nvd.macro_f1.iloc[0]}")
print("Target of FR-23:", "reached" if metrics["meets_target"] else "not reached")

M1 v1, test period 2025-07-01 to 2026-11-14 (110,553 CVEs):
- macro-F1 0.571 with the strict split, 0.605 retrained each quarter (description only: 0.501; indicative target: 0.7)
- accuracy 67.3%; right or one class away 96.8%
- CVEs not scored by their CNA (those M1 predicts): macro-F1 0.482, accuracy 0.645
- for comparison, the CNA and the NVD scoring the same CVE: same class for 56.7% of 72,598 CVEs, macro-F1 0.46
Target of FR-23: not reached


**Interpretation.**

- **What improved since Sprint 1:** the title of the record, the words read by scorer (each CNA has its habits), and above all the second view that predicts the CVSS base metrics and applies the official formula. The comparison table of section 6 gives the gain on the same test period.
- **Why 0.70 is a hard target (section 3):** the people who score do not agree with each other. When the CNA and the NVD score the same CVE they often give different classes: the macro-F1 between these two human experts (printed above) is well below 0.70. Reaching 0.70 would mean that M1 matches a scorer much better than a second expert does.
- **Errors stay close:** almost all predictions are right or one class away. LOW is the weakest class: it is rare (a few percent of the CVEs).
- **Drift:** the quarter-by-quarter table shows the loss of a model that is not retrained (new CNAs, new habits). M1 is retrained each quarter (`train_m1 --activate`), each version keeps its metrics (FR-26).
- **Honest limit:** on the CVEs that their CNA did not score, the ones M1 really predicts, the scores are lower than on the whole test period. This is why a predicted severity is always labeled *predicted* in the console, with its confidence (FR-06), and why the confidence table matters: the most confident predictions are clearly more reliable.

**Decision (risk R4 plan):** M1 predicts the severity only (no CWE target). The four CVSS classes are kept and the result is reported as it is, with the indicative target of FR-23.

In [19]:
conn.close()